## Étape 1 : Préparation du Notebook Jupyter

# 1. Importation des bibliothèques

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import mlflow
import mlflow.sklearn
from pycaret.classification import *

## 2. Chargement du dataset

In [ ]:
# Charger le dataset
df = pd.read_csv("credit_scoring_dataset_v2.csv")

## 3. Exploration des données

In [ ]:
# Aperçu
print(df.head())
print(df.info())
print(df.describe())

## 4. Prétraitement des données
- Numériques : normalisation ou standardisation (Age, Revenu, Montant_Credit…).

- Catégoriques : encodage (OneHotEncoder ou OrdinalEncoder).

- Split train/test : 80/20.

In [ ]:

X = df.drop("Defaut_Paiement", axis=1)
y = df["Defaut_Paiement"]

num_features = ["Age","Revenu","Montant_Credit","Duree_Mois","Score_Credit","Nb_Credits_Prec","Ratio_Dette_Revenu"]
cat_features = ["Statut_Emploi","Niveau_Education","Type_Propriete","Historique_Credit","Objet_Credit"]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## 5. Modélisation classique (Scikit-Learn)
Tester plusieurs modèles : KNN, DecisionTree, RandomForest.

In [ ]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

## 6. Suivi avec MLflow

In [ ]:

mlflow.set_experiment("CreditScoring")

with mlflow.start_run():
    model.fit(X_train, y_train)
    mlflow.sklearn.log_model(model, "model")
    mlflow.log_metric("accuracy", model.score(X_test, y_test))


## 7. Automatisation avec PyCaret

In [ ]:

exp = setup(data=df, target="Defaut_Paiement", session_id=42)
best_model = compare_models()

## Étape 2 : Application pour les utilisateurs

In [ ]:
import streamlit as st
import pickle

# Charger le modèle entraîné
model = pickle.load(open("credit_model.pkl", "rb"))

st.title("Credit Scoring App")

age = st.number_input("Âge", min_value=18, max_value=100)
revenu = st.number_input("Revenu mensuel")
montant = st.number_input("Montant du crédit")
duree = st.number_input("Durée en mois")
score = st.number_input("Score crédit")
nb_prev = st.number_input("Nombre crédits précédents")
ratio = st.number_input("Ratio dette/revenu")

statut = st.selectbox("Statut emploi", ["CDI","CDD","Sans emploi"])
education = st